# Mass Recomputation: Median + Asymmetric Uncertainties

Recomputes stellar masses for `training_stars.csv`, `validation_stars.csv`, and `mmcoeval.csv` using the full Mann et al. (2019) posterior distribution instead of the mean ± std summary.

**What changes:**
- `mass_msun` → **median** of the posterior (previously: mean)
- `mass_msun_err_lo` → median − 16th percentile (new, lower asymmetric uncertainty)
- `mass_msun_err_hi` → 84th percentile − median (new, upper asymmetric uncertainty)
- `mass_msun_err` → **dropped** (old symmetric std column)

**No crossmatching is performed.** All inputs (`k_m_0`, `k_cmsig`, `A_Ks_err`, `parallax`, `parallax_error`) are read directly from the saved CSVs produced by `twomass_xmatch.ipynb`.

**Note on `mk_mass.py`:** The upstream `Mann_2019/mk_mass.py` used `if post == None:` which raises a `ValueError` when a pre-loaded numpy array is passed. This has been patched to `if post is None:` to allow FITS pre-loading.

In [4]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import pkg_resources
from astropy.io import fits

CF_DATA = Path("cf_data")
sys.path.insert(0, "Mann_2019")
import mk_mass

## Load existing catalogs

In [5]:
train = pd.read_csv(CF_DATA / "training_stars.csv",   dtype={"source_id": "Int64"})
val   = pd.read_csv(CF_DATA / "validation_stars.csv", dtype={"source_id": "Int64"})
mm    = pd.read_csv(CF_DATA / "mmcoeval.csv",          dtype={"source_id": "Int64"})

for name, df in [("training_stars", train), ("validation_stars", val), ("mmcoeval", mm)]:
    n_total    = len(df)
    n_km       = df["k_m_0"].notna().sum()
    n_mass     = df["mass_msun"].notna().sum()
    print(f"{name}: {n_total} rows | {n_km} with k_m_0 | {n_mass} with existing mass")

training_stars: 6119 rows | 6119 with k_m_0 | 6119 with existing mass
validation_stars: 228 rows | 228 with k_m_0 | 228 with existing mass
mmcoeval: 34 rows | 34 with k_m_0 | 33 with existing mass


## Pre-load Mann 2019 FITS posterior

The FITS chain (`Mk-M_7_trim.fits`) contains the MCMC posterior for the M_Ks–Mass polynomial coefficients. Pre-loading it once avoids repeated disk reads during the per-star loop.

In [6]:
datapath  = pkg_resources.resource_filename("mk_mass", "resources")
post_data = fits.open(datapath + "/Mk-M_7_trim.fits")[0].data

print(f"FITS posterior loaded: {post_data.shape[0]:,} MCMC samples, {post_data.shape[1]} parameters")

FITS posterior loaded: 400,000 MCMC samples, 7 parameters


## Mass computation using the full posterior

`compute_mass_percentiles` draws the full posterior distribution for a single row and returns the median, lower uncertainty (median − p16), and upper uncertainty (p84 − median). `recompute_mass` applies this to a full DataFrame, dropping the old `mass_msun_err` column.

In [7]:
def compute_mass_percentiles(row, post):
    """Return (median, err_lo, err_hi) from the full Mann 2019 posterior."""
    dist  = 1000.0 / row["parallax"]
    edist = (1000.0 / row["parallax"]**2) * row["parallax_error"]
    # k_cmsig is NaN for ~86 training stars (saturated/bright); treat as 0
    ek_phot = row["k_cmsig"] if pd.notna(row["k_cmsig"]) else 0.0
    ek      = np.sqrt(ek_phot**2 + row["A_Ks_err"]**2)

    samples = mk_mass.posterior(
        K=row["k_m_0"], dist=dist, ek=ek, edist=edist,
        oned=False, silent=True, post=post
    )
    samples = np.asarray(samples)
    valid   = samples[np.isfinite(samples)]
    median  = np.median(valid)
    p16     = np.percentile(valid, 16)
    p84     = np.percentile(valid, 84)
    return median, median - p16, p84 - median


def recompute_mass(df, post):
    df = df.copy()
    for col in ["mass_msun", "mass_msun_err", "mass_msun_err_lo", "mass_msun_err_hi"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    print(f"  Computing masses for {len(df)} rows...")
    results = df.apply(lambda row: compute_mass_percentiles(row, post), axis=1)

    df["mass_msun"]        = [r[0] for r in results]
    df["mass_msun_err_lo"] = [r[1] for r in results]
    df["mass_msun_err_hi"] = [r[2] for r in results]
    return df


print("Functions defined.")

Functions defined.


## Apply to all three catalogs

This is the slow step — each star draws ~1.6M posterior samples. Expect ~1–5 minutes depending on CPU.

In [8]:
print("training_stars...")
train_out = recompute_mass(train, post_data)

print("validation_stars...")
val_out = recompute_mass(val, post_data)

print("mmcoeval...")
mm_out = recompute_mass(mm, post_data)

print()
for name, df_in, df_out in [
    ("training_stars",   train, train_out),
    ("validation_stars", val,   val_out),
    ("mmcoeval",         mm,    mm_out),
]:
    n_old = df_in["mass_msun"].notna().sum()
    n_new = df_out["mass_msun"].notna().sum()
    print(f"{name}: {n_old} → {n_new} stars with mass (should be unchanged)")

training_stars...
  Computing masses for 6119 rows...
0.00% of posterior is outside suggested range for the relation
0.05% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
0.02% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
0.05% of posterior is outside suggested range for the relation
0.03% of posterior is outside suggested range for the relation
0.04% of posterior is outside suggested range for the relation
0.28% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!


/Users/alanxia/Documents/GitHub/cf_xmatch/Mann_2019/mk_mass.py:129: RuntimeWarning: invalid value encountered in log10
  mk       = kmag + 5 - 5*np.log10(distance)


0.01% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
0.01% of posterior is outside suggested range for the relation
0.01% of posterior is outside suggested range for the relation
0.01% of posterior is outside suggested range for the relation
0.03% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
0.14% of posterior is outside suggested range for the relation
0.02% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
0.00% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
0.02% of posterior is outside suggested range for the relation
0.13% of posterior is outside suggested range for the relation
0.13% of posterior 

## Diagnostics: posterior asymmetry

The main motivation for this change is to capture asymmetry in the mass posterior. The cell below checks how asymmetric the posteriors actually are across each catalog.

In [9]:
for name, df in [("training_stars", train_out), ("validation_stars", val_out), ("mmcoeval", mm_out)]:
    sub = df[df["mass_msun"].notna()].copy()
    if len(sub) == 0:
        print(f"{name}: no mass estimates")
        continue

    # Fractional asymmetry: (err_hi - err_lo) / (err_hi + err_lo)
    # 0 = perfectly symmetric, ±1 = fully one-sided
    asym = (sub["mass_msun_err_hi"] - sub["mass_msun_err_lo"]) / \
           (sub["mass_msun_err_hi"] + sub["mass_msun_err_lo"])
    asym = asym.dropna()

    print(f"{name} ({len(sub)} stars):")
    print(f"  mass_msun       median={sub['mass_msun'].median():.4f}  range=[{sub['mass_msun'].min():.4f}, {sub['mass_msun'].max():.4f}]")
    print(f"  mass_msun_err_lo  mean={sub['mass_msun_err_lo'].mean():.4f}  std={sub['mass_msun_err_lo'].std():.4f}")
    print(f"  mass_msun_err_hi  mean={sub['mass_msun_err_hi'].mean():.4f}  std={sub['mass_msun_err_hi'].std():.4f}")
    print(f"  fractional asymmetry  mean={asym.mean():.4f}  |asym|>0.05: {(asym.abs()>0.05).sum()} stars")
    print()

training_stars (6119 stars):
  mass_msun       median=0.4720  range=[0.0981, 0.6739]
  mass_msun_err_lo  mean=0.0180  std=0.0140
  mass_msun_err_hi  mean=0.0177  std=0.0123
  fractional asymmetry  mean=0.0022  |asym|>0.05: 295 stars

validation_stars (228 stars):
  mass_msun       median=0.4626  range=[0.1036, 0.6661]
  mass_msun_err_lo  mean=0.0109  std=0.0055
  mass_msun_err_hi  mean=0.0111  std=0.0057
  fractional asymmetry  mean=0.0056  |asym|>0.05: 0 stars

mmcoeval (34 stars):
  mass_msun       median=0.2606  range=[0.0625, 0.5409]
  mass_msun_err_lo  mean=0.0070  std=0.0026
  mass_msun_err_hi  mean=0.0070  std=0.0024
  fractional asymmetry  mean=-0.0004  |asym|>0.05: 1 stars



## Save updated catalogs

In [10]:
train_out.to_csv(CF_DATA / "training_stars.csv",   index=False)
val_out.to_csv(  CF_DATA / "validation_stars.csv", index=False)
mm_out.to_csv(   CF_DATA / "mmcoeval.csv",          index=False)

print("Saved all three files.")
for name, df in [("training_stars", train_out), ("validation_stars", val_out), ("mmcoeval", mm_out)]:
    mass_cols = [c for c in df.columns if "mass" in c]
    print(f"  {name}: {len(df)} rows | mass columns: {mass_cols}")

Saved all three files.
  training_stars: 6119 rows | mass columns: ['twomass_id', 'mass_msun', 'mass_msun_err_lo', 'mass_msun_err_hi']
  validation_stars: 228 rows | mass columns: ['twomass_id', 'mass_msun', 'mass_msun_err_lo', 'mass_msun_err_hi']
  mmcoeval: 34 rows | mass columns: ['twomass_id', 'mass_msun', 'mass_msun_err_lo', 'mass_msun_err_hi']
